In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

In [2]:
df = pd.read_csv('/kaggle/input/datasets/ijk037/race-data/race_data.csv')

horses = [
    "Shadowfax", "Iron Duke", "Morningstar", "Red Tide",
    "Gallant Fox", "Blue Streak", "Copper Prince", "Last Chance"
]

In [3]:
N_SIM = 100000

win_counts = {horse: 0 for horse in horses}

for _ in range(N_SIM):
    # Sample one historical race (with replacement)
    sample = df.sample(n=1, replace=True).iloc[0]
    
    # Find winner (minimum time)
    times = {horse: sample[horse] for horse in horses}
    winner = min(times, key=times.get)
    
    win_counts[winner] += 1

# Convert to probabilities
p = {horse: win_counts[horse] / N_SIM for horse in horses}

print("Estimated Probabilities (Bootstrap):")
for h in horses:
    print(f"{h}: {p[h]:.4f}")

Estimated Probabilities (Bootstrap):
Shadowfax: 0.3577
Iron Duke: 0.1471
Morningstar: 0.3252
Red Tide: 0.0532
Gallant Fox: 0.0562
Blue Streak: 0.0288
Copper Prince: 0.0208
Last Chance: 0.0109


In [4]:
f = {
    "Shadowfax": 0.08,
    "Iron Duke": 0.09,
    "Morningstar": 0.11,
    "Red Tide": 0.13,
    "Gallant Fox": 0.14,
    "Blue Streak": 0.15,
    "Copper Prince": 0.14,
    "Last Chance": 0.16
}

In [5]:
ev = {}

for horse in horses:
    ev[horse] = p[horse] * (0.85 / f[horse]) - 1

print("\nExpected Value (per £1):")
for h in horses:
    print(f"{h}: {ev[h]:.4f}")


Expected Value (per £1):
Shadowfax: 2.8006
Iron Duke: 0.3896
Morningstar: 1.5133
Red Tide: -0.6523
Gallant Fox: -0.6586
Blue Streak: -0.8369
Copper Prince: -0.8737
Last Chance: -0.9419


In [6]:
bankroll = 10000
fraction = 0.5  # fractional Kelly

bets = {}

for horse in horses:
    edge = p[horse] * (0.85 / f[horse]) - 1
    odds = (0.85 / f[horse]) - 1
    
    if edge > 0:
        kelly = edge / odds
        bets[horse] = bankroll * fraction * kelly
    else:
        bets[horse] = 0

# Normalize if needed
total_bet = sum(bets.values())
if total_bet > bankroll:
    scale = bankroll / total_bet
    for horse in bets:
        bets[horse] *= scale

In [7]:
print("\nRecommended Stakes (£):")
for h in horses:
    print(f"{h}: £{bets[h]:.2f}")

print(f"\nTotal Bet: £{sum(bets.values()):.2f}")


Recommended Stakes (£):
Shadowfax: £1454.84
Iron Duke: £230.66
Morningstar: £1124.75
Red Tide: £0.00
Gallant Fox: £0.00
Blue Streak: £0.00
Copper Prince: £0.00
Last Chance: £0.00

Total Bet: £2810.25
